# Hasbrouck VAR / IRF — Analyse du market impact sur futures index

Ce notebook fournit un pipeline complet et réutilisable pour analyser le **market impact** à partir de trades exécutés sur futures index, par exemple **DAX** et **SXE / Euro Stoxx**.

Il couvre :

1. chargement et nettoyage des données ;
2. construction des variables microstructurelles ;
3. agrégation intraday en buckets ;
4. estimation du VAR de Hasbrouck ;
5. calcul des **IRF** et de l'impact cumulé ;
6. diagnostics économétriques ;
7. VAR enrichi avec spread et imbalance ;
8. segmentation intraday / extraday ;
9. comparaison entre instruments ;
10. fonctions réutilisables pour tes propres datasets.

## Colonnes attendues

Le DataFrame doit contenir au minimum :

| Colonne | Description |
|---|---|
| `timestamp` | horodatage en secondes, millisecondes, ou datetime |
| `price` | prix du trade |
| `qté` | quantité exécutée |
| `sens` | sens agressif du trade : `+1` buy agressif, `-1` sell agressif |
| `mid` | mid-price au moment du trade |
| `bid` | best bid au moment du trade |
| `ask` | best ask au moment du trade |

## Idée du modèle

On construit généralement :

$$
r_t = \frac{mid_t - mid_{t-1}}{ticksize}
$$

et :

$$
x_t = \sum_{i \in bucket\ t} sens_i \log(1 + qté_i)
$$

Puis on estime un VAR :

$$
Y_t = A_1Y_{t-1} + \cdots + A_KY_{t-K} + u_t
$$

avec :

$$
Y_t = \begin{pmatrix} r_t \\ x_t \end{pmatrix}
$$

L'IRF d'intérêt est :

$$
IRF(h) = \frac{\partial r_{t+h}}{\partial \eta_t^x}
$$

et l'impact cumulé :

$$
CI(H) = \sum_{h=0}^{H} IRF(h)
$$

Si $r_t$ est en ticks, alors $CI(H)$ est aussi en ticks.

## 0. Installation des dépendances

Décommente la cellule suivante si tu dois installer les packages.

In [ ]:
# !pip install pandas numpy matplotlib statsmodels scipy scikit-learn

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.api import VAR
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

## 2. Paramètres globaux

À adapter selon tes contrats.

Pour le DAX future, le tick size dépend du contrat utilisé. Par exemple :

- FDAX : tick souvent `0.5` point ;
- FDXM / mini-DAX : à vérifier selon tes données ;
- Euro Stoxx / SXE : adapte à ton contrat exact.

In [ ]:
CONFIG = {
    "timestamp_col": "timestamp",
    "trade_price_col": "price",
    "volume_col": "qté",
    "side_col": "sens",
    "mid_col": "mid",
    "bid_col": "bid",
    "ask_col": "ask",

    # À adapter
    "tick_size": 0.5,

    # Fréquence d'agrégation
    "bucket": "1s",

    # VAR / IRF
    "maxlags": 30,
    "horizon": 60,

    # Transformation du volume : "raw", "log", "sqrt", "power"
    "volume_transform": "log",

    # Standardiser x permet d'interpréter le choc comme 1 écart-type d'orderflow signé.
    "standardize_orderflow": True,
}

## 3. Charger les données

Remplace le chemin par ton fichier.

Exemples :

```python
file_path = "data/dax_trades.csv"
df = pd.read_csv(file_path)
```

ou :

```python
df = pd.read_parquet("data/dax_trades.parquet")
```

In [ ]:
# Exemple CSV
# file_path = "data/dax_trades.csv"
# df = pd.read_csv(file_path)

# Exemple Parquet
# file_path = "data/dax_trades.parquet"
# df = pd.read_parquet(file_path)

# Vérification attendue :
# df.head()

## 4. Fonctions utilitaires de nettoyage

In [ ]:
def convert_timestamp_to_datetime(series: pd.Series) -> pd.Series:
    """
    Convertit une colonne timestamp en datetime.

    Cas gérés :
    - datetime déjà existant ;
    - timestamp Unix en millisecondes ;
    - timestamp Unix en secondes ;
    - secondes relatives depuis le début de session ;
    - string datetime.
    """
    if np.issubdtype(series.dtype, np.datetime64):
        return pd.to_datetime(series)

    if np.issubdtype(series.dtype, np.number):
        median_ts = series.dropna().median()

        # Timestamp Unix en millisecondes : environ 1e12 ou plus.
        if median_ts > 1e12:
            return pd.to_datetime(series, unit="ms")

        # Timestamp Unix en secondes : environ 1e9.
        if median_ts > 1e9:
            return pd.to_datetime(series, unit="s")

        # Sinon, on suppose des secondes relatives depuis le début de session.
        return pd.Timestamp("2000-01-01") + pd.to_timedelta(series, unit="s")

    return pd.to_datetime(series)


def map_side_column(df: pd.DataFrame, side_col: str = "sens") -> pd.DataFrame:
    """
    Convertit la colonne de sens en +1 / -1 si elle est textuelle.

    Si la colonne est déjà numérique avec +1 / -1, elle est conservée.
    """
    out = df.copy()

    if side_col not in out.columns:
        raise ValueError(f"Colonne manquante : {side_col}")

    if np.issubdtype(out[side_col].dtype, np.number):
        out[side_col] = pd.to_numeric(out[side_col], errors="coerce")
        return out

    mapping = {
        "BUY": 1, "Buy": 1, "buy": 1, "B": 1, "b": 1,
        "ACHAT": 1, "Achat": 1, "achat": 1,
        "ASK": 1, "ask": 1,  # parfois trade à l'ask = achat agressif

        "SELL": -1, "Sell": -1, "sell": -1, "S": -1, "s": -1,
        "VENTE": -1, "Vente": -1, "vente": -1,
        "BID": -1, "bid": -1,  # parfois trade au bid = vente agressive
    }

    out[side_col] = out[side_col].map(mapping)
    return out


def clean_trade_dataframe(
    df: pd.DataFrame,
    timestamp_col="timestamp",
    trade_price_col="price",
    volume_col="qté",
    side_col="sens",
    mid_col="mid",
    bid_col="bid",
    ask_col="ask",
) -> pd.DataFrame:
    """
    Nettoyage minimal pour un dataset de trades + quotes.
    """
    required = [timestamp_col, trade_price_col, volume_col, side_col, mid_col, bid_col, ask_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes : {missing}")

    out = df[required].copy()
    out = map_side_column(out, side_col=side_col)

    out[timestamp_col] = convert_timestamp_to_datetime(out[timestamp_col])
    out = out.set_index(timestamp_col).sort_index()

    for c in [trade_price_col, volume_col, side_col, mid_col, bid_col, ask_col]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.dropna()
    out = out[out[side_col].isin([-1, 1])]
    out = out[out[volume_col] > 0]
    out = out[out[ask_col] >= out[bid_col]]
    out = out[out[mid_col] > 0]
    out = out[out[trade_price_col] > 0]

    return out

## 5. Fonctions de construction des variables microstructurelles

On construit :

- spread ;
- spread en ticks ;
- orderflow signé trade par trade ;
- rendements du mid en ticks ;
- imbalance de volume ;
- trade count ;
- volume total ;
- variables bucketisées.

In [ ]:
def build_signed_orderflow(volume: pd.Series, side: pd.Series, transform: str = "log", psi: float = 0.5) -> pd.Series:
    """
    Construit l'orderflow signé trade-level.

    transform:
    - raw  : side * volume
    - log  : side * log(1 + volume)
    - sqrt : side * sqrt(volume)
    - power: side * volume ** psi
    """
    if transform == "raw":
        return side * volume
    if transform == "log":
        return side * np.log1p(volume)
    if transform == "sqrt":
        return side * np.sqrt(volume)
    if transform == "power":
        return side * (volume ** psi)
    raise ValueError("transform doit être 'raw', 'log', 'sqrt' ou 'power'")


def build_bucket_data(
    df_clean: pd.DataFrame,
    trade_price_col="price",
    volume_col="qté",
    side_col="sens",
    mid_col="mid",
    bid_col="bid",
    ask_col="ask",
    tick_size=0.5,
    bucket="1s",
    volume_transform="log",
    use_mid_returns=True,
    psi=0.5,
    drop_zero_return_buckets=False,
) -> pd.DataFrame:
    """
    Agrège les trades en buckets réguliers.

    La variable r est la variation du mid en ticks.
    La variable x est l'orderflow signé agrégé.
    """
    df = df_clean.copy()

    df["spread"] = df[ask_col] - df[bid_col]
    df["spread_ticks"] = df["spread"] / tick_size
    df["x_trade"] = build_signed_orderflow(
        volume=df[volume_col],
        side=df[side_col],
        transform=volume_transform,
        psi=psi,
    )

    if use_mid_returns:
        reference_price = df[mid_col].resample(bucket).last().ffill()
    else:
        reference_price = df[trade_price_col].resample(bucket).last().ffill()

    r_bucket = reference_price.diff() / tick_size
    x_bucket = df["x_trade"].resample(bucket).sum()
    volume_bucket = df[volume_col].resample(bucket).sum()
    trade_count_bucket = df[volume_col].resample(bucket).count()

    buy_volume = df.loc[df[side_col] == 1, volume_col].resample(bucket).sum()
    sell_volume = df.loc[df[side_col] == -1, volume_col].resample(bucket).sum()
    buy_volume = buy_volume.reindex(x_bucket.index).fillna(0.0)
    sell_volume = sell_volume.reindex(x_bucket.index).fillna(0.0)

    imbalance = (buy_volume - sell_volume) / (buy_volume + sell_volume)
    imbalance = imbalance.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    spread_bucket = df["spread_ticks"].resample(bucket).mean().ffill()
    mid_bucket = reference_price

    out = pd.DataFrame({
        "r": r_bucket,
        "x": x_bucket,
        "volume": volume_bucket,
        "trade_count": trade_count_bucket,
        "buy_volume": buy_volume,
        "sell_volume": sell_volume,
        "imbalance": imbalance,
        "spread_ticks": spread_bucket,
        "mid": mid_bucket,
    }).dropna()

    if drop_zero_return_buckets:
        out = out[out["r"] != 0]

    return out

## 6. Préparer les variables du VAR

On garde généralement `r` en ticks pour que l'impact cumulé soit interprétable en ticks.

Pour `x`, deux choix :

1. seulement centrer `x` : le choc correspond à une unité de la transformation choisie ;
2. centrer et standardiser `x` : le choc correspond à **1 écart-type d'orderflow signé**.

La deuxième option est souvent plus lisible pour comparer plusieurs contrats.

In [ ]:
def prepare_var_data(
    bucket_data: pd.DataFrame,
    variables=("r", "x"),
    keep_r_in_ticks=True,
    standardize_non_r=True,
) -> pd.DataFrame:
    """
    Prépare les variables d'un VAR : centrage et standardisation optionnelle.

    Par défaut :
    - r reste en ticks, seulement centré ;
    - les autres variables sont centrées et réduites.
    """
    var_data = bucket_data[list(variables)].dropna().copy()

    for col in var_data.columns:
        var_data[col] = var_data[col] - var_data[col].mean()

    if standardize_non_r:
        for col in var_data.columns:
            if keep_r_in_ticks and col == "r":
                continue
            std = var_data[col].std()
            if std > 0 and not np.isnan(std):
                var_data[col] = var_data[col] / std
            else:
                raise ValueError(f"Variance nulle ou invalide pour la variable {col}")

    return var_data

## 7. Estimation VAR et calcul de l'IRF

In [ ]:
def fit_var_model(var_data: pd.DataFrame, maxlags=30, ic="aic"):
    """
    Estime un VAR avec sélection automatique du lag par AIC/BIC/HQIC/FPE.
    """
    model = VAR(var_data)
    results = model.fit(maxlags=maxlags, ic=ic)
    return results


def compute_irf_to_orderflow(
    results,
    horizon=60,
    response_var="r",
    impulse_var="x",
):
    """
    Calcule l'IRF de response_var à un choc sur impulse_var.

    Retourne :
    - irf object statsmodels ;
    - réponse horizon par horizon ;
    - impact cumulé.
    """
    irf = results.irf(horizon)
    names = results.names

    response_idx = names.index(response_var)
    impulse_idx = names.index(impulse_var)

    response = irf.irfs[:, response_idx, impulse_idx]
    cumulative = np.cumsum(response)

    return irf, response, cumulative


def plot_irf_and_cumulative(response, cumulative, title_prefix="Hasbrouck VAR", ylabel_unit="ticks"):
    h = np.arange(len(response))

    plt.figure(figsize=(10, 5))
    plt.plot(h, response, marker="o")
    plt.axhline(0, linestyle="--")
    plt.xlabel("Horizon en buckets")
    plt.ylabel(f"IRF en {ylabel_unit}")
    plt.title(f"{title_prefix} — IRF : réponse de r à un choc sur x")
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(h, cumulative, marker="o")
    plt.axhline(0, linestyle="--")
    plt.xlabel("Horizon en buckets")
    plt.ylabel(f"Impact cumulé en {ylabel_unit}")
    plt.title(f"{title_prefix} — Impact cumulé")
    plt.show()

## 8. Pipeline complet : VAR simple de Hasbrouck

Cette fonction fait tout :

1. nettoyage ;
2. bucketisation ;
3. préparation du VAR ;
4. estimation ;
5. IRF ;
6. impact cumulé.

In [ ]:
def run_hasbrouck_pipeline(
    df,
    timestamp_col="timestamp",
    trade_price_col="price",
    volume_col="qté",
    side_col="sens",
    mid_col="mid",
    bid_col="bid",
    ask_col="ask",
    tick_size=0.5,
    bucket="1s",
    maxlags=30,
    horizon=60,
    volume_transform="log",
    psi=0.5,
    use_mid_returns=True,
    variables=("r", "x"),
    standardize_non_r=True,
    ic="aic",
    plot=True,
):
    df_clean = clean_trade_dataframe(
        df,
        timestamp_col=timestamp_col,
        trade_price_col=trade_price_col,
        volume_col=volume_col,
        side_col=side_col,
        mid_col=mid_col,
        bid_col=bid_col,
        ask_col=ask_col,
    )

    bucket_data = build_bucket_data(
        df_clean,
        trade_price_col=trade_price_col,
        volume_col=volume_col,
        side_col=side_col,
        mid_col=mid_col,
        bid_col=bid_col,
        ask_col=ask_col,
        tick_size=tick_size,
        bucket=bucket,
        volume_transform=volume_transform,
        use_mid_returns=use_mid_returns,
        psi=psi,
    )

    var_data = prepare_var_data(
        bucket_data,
        variables=variables,
        keep_r_in_ticks=True,
        standardize_non_r=standardize_non_r,
    )

    results = fit_var_model(var_data, maxlags=maxlags, ic=ic)
    irf, response, cumulative = compute_irf_to_orderflow(
        results,
        horizon=horizon,
        response_var="r",
        impulse_var="x",
    )

    if plot:
        plot_irf_and_cumulative(response, cumulative, title_prefix="Hasbrouck VAR")

    return {
        "df_clean": df_clean,
        "bucket_data": bucket_data,
        "var_data": var_data,
        "results": results,
        "irf_object": irf,
        "response_r_to_x": response,
        "cumulative_impact": cumulative,
        "selected_lags": results.k_ar,
        "config": {
            "tick_size": tick_size,
            "bucket": bucket,
            "maxlags": maxlags,
            "horizon": horizon,
            "volume_transform": volume_transform,
            "standardize_non_r": standardize_non_r,
            "variables": variables,
        }
    }

## 9. Exemple d'utilisation sur ton DataFrame

Décommente lorsque `df` est chargé.

In [ ]:
# out = run_hasbrouck_pipeline(
#     df=df,
#     timestamp_col="timestamp",
#     trade_price_col="price",
#     volume_col="qté",
#     side_col="sens",
#     mid_col="mid",
#     bid_col="bid",
#     ask_col="ask",
#     tick_size=CONFIG["tick_size"],
#     bucket=CONFIG["bucket"],
#     maxlags=CONFIG["maxlags"],
#     horizon=CONFIG["horizon"],
#     volume_transform=CONFIG["volume_transform"],
#     standardize_non_r=CONFIG["standardize_orderflow"],
#     variables=("r", "x"),
#     plot=True,
# )
#
# print("Lags sélectionnés :", out["selected_lags"])
# print("Impact cumulé final :", out["cumulative_impact"][-1], "ticks")

## 10. Diagnostics du VAR

À regarder systématiquement :

1. stabilité du VAR ;
2. autocorrélation des résidus ;
3. résumé des coefficients ;
4. robustesse au choix du lag ;
5. robustesse au bucket.

In [ ]:
def var_diagnostics(results):
    print("========== Résumé VAR ==========")
    print(results.summary())

    print("\n========== Stabilité ==========")
    print("VAR stable :", results.is_stable())

    print("\n========== Test de blancheur des résidus ==========")
    try:
        print(results.test_whiteness())
    except Exception as e:
        print("Test de blancheur indisponible :", e)

    print("\n========== Test de normalité des résidus ==========")
    try:
        print(results.test_normality())
    except Exception as e:
        print("Test de normalité indisponible :", e)


def plot_var_residuals(results):
    resid = results.resid
    resid.plot(figsize=(12, 6), title="Résidus du VAR")
    plt.axhline(0, linestyle="--")
    plt.show()

# Exemple :
# var_diagnostics(out["results"])
# plot_var_residuals(out["results"])

## 11. VAR enrichi avec liquidité : spread et imbalance

Le VAR simple utilise seulement :

$$
Y_t = (r_t, x_t)
$$

On peut enrichir avec :

$$
Y_t = (r_t, x_t, imbalance_t, spread_t)
$$

Cela contrôle partiellement l'état de liquidité.

Attention : ce n'est pas encore un vrai modèle LOB complet, car il manque profondeur et annulations.

In [ ]:
def run_liquidity_var_from_pipeline_output(
    out,
    variables=("r", "x", "imbalance", "spread_ticks"),
    maxlags=30,
    horizon=60,
    ic="aic",
    standardize_non_r=True,
    plot=True,
):
    bucket_data = out["bucket_data"]

    var_data = prepare_var_data(
        bucket_data,
        variables=variables,
        keep_r_in_ticks=True,
        standardize_non_r=standardize_non_r,
    )

    results = fit_var_model(var_data, maxlags=maxlags, ic=ic)
    irf, response, cumulative = compute_irf_to_orderflow(
        results,
        horizon=horizon,
        response_var="r",
        impulse_var="x",
    )

    if plot:
        plot_irf_and_cumulative(response, cumulative, title_prefix="VAR enrichi liquidité")

    return {
        "var_data": var_data,
        "results": results,
        "irf_object": irf,
        "response_r_to_x": response,
        "cumulative_impact": cumulative,
        "selected_lags": results.k_ar,
        "variables": variables,
    }

# Exemple :
# out_liq = run_liquidity_var_from_pipeline_output(
#     out,
#     variables=("r", "x", "imbalance", "spread_ticks"),
#     maxlags=30,
#     horizon=60,
#     plot=True,
# )
# print("Impact cumulé final, VAR enrichi :", out_liq["cumulative_impact"][-1], "ticks")

## 12. Analyse descriptive intraday

Avant d'interpréter le VAR, il faut comprendre les régimes intraday.

Variables descriptives utiles :

- volume par bucket ;
- nombre de trades ;
- spread moyen ;
- imbalance ;
- realized volatility ;
- saisonnalité par heure.

In [ ]:
def add_realized_volatility(bucket_data: pd.DataFrame, window=60) -> pd.DataFrame:
    """
    Ajoute une volatilité réalisée rolling basée sur r en ticks.

    Si bucket = 1s et window = 60, RV sur 60 secondes.
    """
    out = bucket_data.copy()
    out["rv"] = out["r"].pow(2).rolling(window).sum()
    out["abs_r"] = out["r"].abs()
    return out


def plot_intraday_profiles(bucket_data: pd.DataFrame, freq="5min"):
    """
    Trace des profils intraday simples.
    """
    bd = bucket_data.copy()

    profiles = pd.DataFrame({
        "volume": bd["volume"].resample(freq).sum(),
        "trade_count": bd["trade_count"].resample(freq).sum(),
        "spread_ticks": bd["spread_ticks"].resample(freq).mean(),
        "imbalance": bd["imbalance"].resample(freq).mean(),
        "rv": bd["r"].pow(2).resample(freq).sum(),
    }).dropna()

    for col in profiles.columns:
        plt.figure(figsize=(12, 4))
        plt.plot(profiles.index, profiles[col])
        plt.title(f"Profil intraday — {col}")
        plt.xlabel("Temps")
        plt.ylabel(col)
        plt.show()

    return profiles

# Exemple :
# bd = add_realized_volatility(out["bucket_data"], window=60)
# profiles = plot_intraday_profiles(bd, freq="5min")

## 13. Segmentation intraday / extraday

On distingue :

- **intraday** : dynamique à l'intérieur de la séance, en secondes/minutes ;
- **extraday** : comparaison entre jours, régimes, sessions, contrats, périodes de roll.

Dans la pratique, il est souvent préférable d'estimer des VAR séparés par segment.

In [ ]:
def assign_time_of_day_segment(index: pd.DatetimeIndex) -> pd.Series:
    """
    Segmentation générique intraday.

    À adapter selon horaires exacts Eurex / ton univers.
    """
    hours = index.hour + index.minute / 60.0 + index.second / 3600.0

    seg = pd.Series(index=index, dtype="object")
    seg[(hours >= 7.0) & (hours < 9.0)] = "early_europe"
    seg[(hours >= 9.0) & (hours < 11.5)] = "europe_open"
    seg[(hours >= 11.5) & (hours < 14.5)] = "midday"
    seg[(hours >= 14.5) & (hours < 17.5)] = "us_overlap"
    seg[(hours >= 17.5) & (hours < 22.0)] = "late_session"
    seg[seg.isna()] = "overnight_or_other"
    return seg


def run_segmented_var(
    out,
    segment_col=None,
    min_obs=500,
    variables=("r", "x"),
    maxlags=30,
    horizon=60,
    standardize_non_r=True,
):
    """
    Estime un VAR séparé par segment.

    Si segment_col est None, crée une segmentation time-of-day générique.
    """
    bd = out["bucket_data"].copy()

    if segment_col is None:
        bd["segment"] = assign_time_of_day_segment(bd.index)
    else:
        bd["segment"] = bd[segment_col]

    results_by_segment = {}

    for segment, sub in bd.groupby("segment"):
        sub = sub.dropna()
        if len(sub) < min_obs:
            print(f"Segment ignoré : {segment}, observations insuffisantes : {len(sub)}")
            continue

        try:
            var_data = prepare_var_data(
                sub,
                variables=variables,
                keep_r_in_ticks=True,
                standardize_non_r=standardize_non_r,
            )
            res = fit_var_model(var_data, maxlags=maxlags, ic="aic")
            irf, response, cumulative = compute_irf_to_orderflow(res, horizon=horizon)

            results_by_segment[segment] = {
                "data": sub,
                "var_data": var_data,
                "results": res,
                "response_r_to_x": response,
                "cumulative_impact": cumulative,
                "selected_lags": res.k_ar,
            }
            print(f"Segment OK : {segment}, n={len(sub)}, lags={res.k_ar}, CI_final={cumulative[-1]:.4f}")

        except Exception as e:
            print(f"Erreur segment {segment} : {e}")

    return results_by_segment


def plot_segmented_cumulative_impacts(results_by_segment):
    plt.figure(figsize=(12, 6))
    for segment, obj in results_by_segment.items():
        plt.plot(obj["cumulative_impact"], label=segment)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Horizon en buckets")
    plt.ylabel("Impact cumulé en ticks")
    plt.title("Impact cumulé par segment")
    plt.legend()
    plt.show()

# Exemple :
# seg_results = run_segmented_var(out, min_obs=500, variables=("r", "x"))
# plot_segmented_cumulative_impacts(seg_results)

## 14. Analyse par jour : extraday

Une analyse extraday utile consiste à estimer l'impact final par jour :

$$
CI_d(H)
$$

puis à analyser sa distribution entre jours.

Cela permet de voir si l'impact est stable ou si quelques jours de stress dominent.

In [ ]:
def run_daily_var_impacts(
    out,
    min_obs_per_day=500,
    variables=("r", "x"),
    maxlags=30,
    horizon=60,
    standardize_non_r=True,
):
    bd = out["bucket_data"].copy()
    bd["date"] = bd.index.date

    daily_results = []
    objects = {}

    for date, sub in bd.groupby("date"):
        if len(sub) < min_obs_per_day:
            continue

        try:
            var_data = prepare_var_data(sub, variables=variables, keep_r_in_ticks=True, standardize_non_r=standardize_non_r)
            res = fit_var_model(var_data, maxlags=maxlags, ic="aic")
            irf, response, cumulative = compute_irf_to_orderflow(res, horizon=horizon)

            row = {
                "date": pd.to_datetime(date),
                "n_obs": len(sub),
                "selected_lags": res.k_ar,
                "ci_final": cumulative[-1],
                "ci_max": np.max(cumulative),
                "ci_min": np.min(cumulative),
                "volume": sub["volume"].sum(),
                "rv": sub["r"].pow(2).sum(),
                "mean_spread_ticks": sub["spread_ticks"].mean(),
            }
            daily_results.append(row)
            objects[date] = {
                "results": res,
                "response_r_to_x": response,
                "cumulative_impact": cumulative,
            }
        except Exception as e:
            print(f"Erreur jour {date}: {e}")

    daily_df = pd.DataFrame(daily_results).sort_values("date") if daily_results else pd.DataFrame()
    return daily_df, objects


def plot_daily_impact_summary(daily_df):
    if daily_df.empty:
        print("Aucun résultat journalier.")
        return

    plt.figure(figsize=(12, 4))
    plt.plot(daily_df["date"], daily_df["ci_final"], marker="o")
    plt.axhline(0, linestyle="--")
    plt.title("Impact cumulé final par jour")
    plt.xlabel("Date")
    plt.ylabel("CI final en ticks")
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.hist(daily_df["ci_final"].dropna(), bins=30)
    plt.axvline(0, linestyle="--")
    plt.title("Distribution de l'impact cumulé final par jour")
    plt.xlabel("CI final en ticks")
    plt.ylabel("Fréquence")
    plt.show()

# Exemple :
# daily_df, daily_objects = run_daily_var_impacts(out, min_obs_per_day=500)
# display(daily_df.head())
# plot_daily_impact_summary(daily_df)

## 15. Comparaison entre deux instruments : DAX vs SXE

Tu peux charger deux DataFrames séparés, par exemple `df_dax` et `df_sxe`, puis exécuter le même pipeline.

Important : pour comparer proprement, il faut idéalement comparer :

- impact en ticks ;
- impact en bps ;
- impact normalisé par volatilité ;
- impact par choc standardisé d'orderflow.

In [ ]:
def compare_two_instruments(
    df_a,
    df_b,
    name_a="DAX",
    name_b="SXE",
    tick_size_a=0.5,
    tick_size_b=1.0,
    bucket="1s",
    maxlags=30,
    horizon=60,
    volume_transform="log",
):
    out_a = run_hasbrouck_pipeline(
        df_a,
        tick_size=tick_size_a,
        bucket=bucket,
        maxlags=maxlags,
        horizon=horizon,
        volume_transform=volume_transform,
        plot=False,
    )

    out_b = run_hasbrouck_pipeline(
        df_b,
        tick_size=tick_size_b,
        bucket=bucket,
        maxlags=maxlags,
        horizon=horizon,
        volume_transform=volume_transform,
        plot=False,
    )

    plt.figure(figsize=(12, 6))
    plt.plot(out_a["cumulative_impact"], label=f"{name_a} — CI")
    plt.plot(out_b["cumulative_impact"], label=f"{name_b} — CI")
    plt.axhline(0, linestyle="--")
    plt.xlabel("Horizon en buckets")
    plt.ylabel("Impact cumulé en ticks")
    plt.title("Comparaison d'impact cumulé")
    plt.legend()
    plt.show()

    summary = pd.DataFrame({
        "instrument": [name_a, name_b],
        "selected_lags": [out_a["selected_lags"], out_b["selected_lags"]],
        "ci_final_ticks": [out_a["cumulative_impact"][-1], out_b["cumulative_impact"][-1]],
        "ci_max_ticks": [np.max(out_a["cumulative_impact"]), np.max(out_b["cumulative_impact"])],
        "ci_min_ticks": [np.min(out_a["cumulative_impact"]), np.min(out_b["cumulative_impact"])],
    })

    return out_a, out_b, summary

# Exemple :
# out_dax, out_sxe, comparison = compare_two_instruments(
#     df_dax,
#     df_sxe,
#     name_a="DAX",
#     name_b="SXE",
#     tick_size_a=0.5,
#     tick_size_b=1.0,
# )
# display(comparison)

## 16. Analyse de sensibilité : bucket, horizon, transformation du volume

Il est important de tester la robustesse de l'impact aux choix de modélisation.

Variables sensibles :

- `bucket` : `250ms`, `500ms`, `1s`, `5s` ;
- `volume_transform` : `raw`, `log`, `sqrt` ;
- `maxlags` ;
- segmentation intraday ;
- inclusion de spread/imbalance.

In [ ]:
def sensitivity_analysis(
    df,
    tick_size=0.5,
    buckets=("500ms", "1s", "5s"),
    transforms=("raw", "log", "sqrt"),
    maxlags=30,
    horizon=60,
):
    rows = []
    objects = {}

    for bucket in buckets:
        for transform in transforms:
            key = f"bucket={bucket}|transform={transform}"
            try:
                out = run_hasbrouck_pipeline(
                    df,
                    tick_size=tick_size,
                    bucket=bucket,
                    maxlags=maxlags,
                    horizon=horizon,
                    volume_transform=transform,
                    plot=False,
                )
                rows.append({
                    "bucket": bucket,
                    "transform": transform,
                    "selected_lags": out["selected_lags"],
                    "ci_final": out["cumulative_impact"][-1],
                    "ci_max": np.max(out["cumulative_impact"]),
                    "ci_min": np.min(out["cumulative_impact"]),
                    "n_obs": len(out["var_data"]),
                })
                objects[key] = out
                print("OK", key)
            except Exception as e:
                print("Erreur", key, e)

    result_df = pd.DataFrame(rows)
    return result_df, objects

# Exemple :
# sens_df, sens_objects = sensitivity_analysis(
#     df,
#     tick_size=0.5,
#     buckets=("500ms", "1s", "5s"),
#     transforms=("raw", "log", "sqrt"),
# )
# display(sens_df)

## 17. Event study simple autour des trades

Le VAR mesure l'impact dynamique via innovations. Une analyse complémentaire consiste à mesurer directement l'impact moyen après les trades.

Pour chaque trade :

$$
MI_i(\tau) = sens_i \cdot \frac{mid_{t_i+\tau} - mid_{t_i^-}}{ticksize}
$$

Cela donne une mesure descriptive de l'impact réalisé.

In [ ]:
def event_study_trade_impact(
    df_clean,
    horizons=("1s", "5s", "30s", "60s"),
    volume_col="qté",
    side_col="sens",
    mid_col="mid",
    tick_size=0.5,
    volume_bins=5,
):
    """
    Event study trade-level : impact signé moyen à plusieurs horizons.

    Fonction simple et robuste, mais potentiellement coûteuse sur très gros dataset.
    """
    df = df_clean.copy().sort_index()
    mid_series = df[mid_col].copy()

    rows = []
    for horizon in horizons:
        future_index = df.index + pd.to_timedelta(horizon)

        # Reindex par nearest forward : mid au temps t + horizon.
        future_mid = mid_series.reindex(future_index, method="ffill")
        future_mid.index = df.index

        impact = df[side_col] * (future_mid - df[mid_col]) / tick_size
        tmp = pd.DataFrame({
            "impact": impact,
            "volume": df[volume_col],
            "side": df[side_col],
            "horizon": horizon,
        }).dropna()

        tmp["volume_bin"] = pd.qcut(tmp["volume"], q=volume_bins, duplicates="drop")
        rows.append(tmp)

    all_impacts = pd.concat(rows)

    summary = (
        all_impacts
        .groupby(["horizon", "volume_bin"])
        .agg(
            mean_impact=("impact", "mean"),
            median_impact=("impact", "median"),
            n=("impact", "size"),
            mean_volume=("volume", "mean"),
        )
        .reset_index()
    )

    return all_impacts, summary


def plot_event_study_summary(summary):
    for horizon, sub in summary.groupby("horizon"):
        plt.figure(figsize=(10, 4))
        plt.plot(sub["mean_volume"], sub["mean_impact"], marker="o")
        plt.axhline(0, linestyle="--")
        plt.xlabel("Volume moyen du bin")
        plt.ylabel("Impact signé moyen en ticks")
        plt.title(f"Event study — Impact par taille, horizon {horizon}")
        plt.show()

# Exemple :
# all_impacts, impact_summary = event_study_trade_impact(
#     out["df_clean"],
#     horizons=("1s", "5s", "30s", "60s"),
#     tick_size=0.5,
# )
# display(impact_summary.head())
# plot_event_study_summary(impact_summary)

## 18. Tester la concavité de l'impact en taille

On teste une relation :

$$
I(Q) = a Q^\psi
$$

soit en log-log :

$$
\log I(Q) = \log a + \psi \log Q + \varepsilon
$$

Il faut l'appliquer sur des impacts moyens par bins, pas directement sur chaque trade, car les impacts individuels peuvent être nuls ou négatifs.

In [ ]:
def estimate_concavity_from_event_summary(summary, horizon="30s"):
    sub = summary[summary["horizon"] == horizon].copy()
    sub = sub[sub["mean_impact"] > 0]
    sub = sub[sub["mean_volume"] > 0]

    if len(sub) < 3:
        raise ValueError("Pas assez de bins positifs pour estimer la concavité.")

    x = np.log(sub["mean_volume"].values)
    y = np.log(sub["mean_impact"].values)

    beta = np.polyfit(x, y, deg=1)
    psi = beta[0]
    log_a = beta[1]

    y_hat = log_a + psi * x

    plt.figure(figsize=(8, 5))
    plt.scatter(x, y, label="Bins")
    plt.plot(x, y_hat, label=f"Fit: psi={psi:.3f}")
    plt.xlabel("log(volume moyen)")
    plt.ylabel("log(impact moyen)")
    plt.title(f"Concavité de l'impact — horizon {horizon}")
    plt.legend()
    plt.show()

    return {"psi": psi, "log_a": log_a, "a": np.exp(log_a), "data": sub}

# Exemple :
# concavity = estimate_concavity_from_event_summary(impact_summary, horizon="30s")
# print(concavity)

## 19. Export des résultats

Tu peux sauvegarder les résultats importants :

- bucket data ;
- IRF ;
- impact cumulé ;
- diagnostics ;
- daily impact summary.

In [ ]:
def export_irf_results(out, path_prefix="hasbrouck_results"):
    h = np.arange(len(out["response_r_to_x"]))
    irf_df = pd.DataFrame({
        "horizon": h,
        "irf_r_to_x": out["response_r_to_x"],
        "cumulative_impact": out["cumulative_impact"],
    })

    irf_path = f"{path_prefix}_irf.csv"
    bucket_path = f"{path_prefix}_bucket_data.csv"
    var_path = f"{path_prefix}_var_data.csv"

    irf_df.to_csv(irf_path, index=False)
    out["bucket_data"].to_csv(bucket_path)
    out["var_data"].to_csv(var_path)

    print("Fichiers exportés :")
    print(irf_path)
    print(bucket_path)
    print(var_path)

    return irf_df

# Exemple :
# irf_df = export_irf_results(out, path_prefix="dax_hasbrouck")

## 20. Lecture économique des sorties

### IRF

$$
IRF(h) = \frac{\partial r_{t+h}}{\partial \eta_t^x}
$$

Si l'IRF est positive après un choc positif d'orderflow, alors l'achat agressif inattendu fait monter le mid-price.

### Impact cumulé

$$
CI(H) = \sum_{h=0}^{H} IRF(h)
$$

Si `r` est en ticks, `CI(H)` est aussi en ticks.

### Interprétation pratique

Si :

```python
out["cumulative_impact"][-1] = 0.4
```

alors un choc positif de 1 écart-type d'orderflow signé déplace le mid d'environ **0.4 tick** à l'horizon choisi.

Pour un market maker, cela peut être interprété comme une approximation du coût d'adverse selection associé à un flux agressif inattendu.

### Attention

Cette interprétation est prédictive, pas strictement causale. Le VAR attribue au trade une dynamique de prix conditionnelle au modèle, mais il ne contrôle pas forcément :

- news macro ;
- carnet complet ;
- profondeur ;
- annulations ;
- queue position ;
- signaux cross-asset.

## 21. Checklist de production

Avant d'utiliser les résultats pour du market making réel :

1. vérifier la qualité du `sens` agressif ;
2. travailler avec le `mid`, pas seulement le prix de trade ;
3. tester plusieurs buckets ;
4. tester plusieurs transformations de volume ;
5. estimer par segments intraday ;
6. exclure ou isoler les périodes de news ;
7. isoler le roll ;
8. comparer VAR simple et VAR enrichi ;
9. vérifier la stabilité du VAR ;
10. contrôler l'autocorrélation résiduelle ;
11. comparer DAX/SXE en ticks, bps et volatilité normalisée ;
12. ne pas interpréter l'IRF comme causalité pure sans prudence.